In [8]:
!git clone https://github.com/emanhamed/Houses-dataset

fatal: destination path 'Houses-dataset' already exists and is not an empty directory.


In [9]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelBinarizer
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Input, concatenate
from tensorflow.keras.optimizers import Adam

print("Dataset cloned and libraries imported successfully!")

Dataset cloned and libraries imported successfully!


In [10]:
df_path = "Houses-dataset/Houses Dataset/HousesInfo.txt"

cols = ["bedrooms", "bathrooms", "area", "zipcode", "price"]
df = pd.read_csv(df_path, sep=" ", header=None, names=cols)

zipcodes = df["zipcode"].value_counts().keys().tolist()
for z in zipcodes:
    if df["zipcode"].value_counts()[z] < 25:
        df = df[df["zipcode"] != z]

continuous = ["bedrooms", "bathrooms", "area"]
cs = MinMaxScaler()
trainContinuous = cs.fit_transform(df[continuous])

zipBinarizer = LabelBinarizer().fit(df["zipcode"])
trainCategorical = zipBinarizer.transform(df["zipcode"])

tabular_features = np.hstack([trainCategorical, trainContinuous])
labels = df["price"].values

print(f"Tabular features preprocessed shape: {tabular_features.shape}")

Tabular features preprocessed shape: (362, 10)


In [11]:
images = []

for i in df.index:
    image_path = f"Houses-dataset/Houses Dataset/{i+1}_frontal.jpg"

    if os.path.exists(image_path):
        image = cv2.imread(image_path)
        image = cv2.resize(image, (64, 64))
        images.append(image)
    else:
        print(f"Warning: Image {image_path} missing.")

image_features = np.array(images) / 255.0

print(f"Image features preprocessed shape: {image_features.shape}")

Image features preprocessed shape: (362, 64, 64, 3)


In [12]:
(trainTab, testTab, trainImg, testImg, trainY, testY) = train_test_split(
    tabular_features, image_features, labels, test_size=0.25, random_state=42
)

print(f"Training samples: {len(trainY)}, Testing samples: {len(testY)}")

Training samples: 271, Testing samples: 91


In [13]:
mlp_input = Input(shape=(trainTab.shape[1],))
x = Dense(16, activation="relu")(mlp_input)
x = Dense(4, activation="relu")(x)
mlp_branch = Model(inputs=mlp_input, outputs=x)

cnn_input = Input(shape=(64, 64, 3))
y = Conv2D(16, (3, 3), padding="same", activation="relu")(cnn_input)
y = MaxPooling2D(pool_size=(2, 2))(y)
y = Conv2D(32, (3, 3), padding="same", activation="relu")(y)
y = MaxPooling2D(pool_size=(2, 2))(y)
y = Flatten()(y)
y = Dense(4, activation="relu")(y)
cnn_branch = Model(inputs=cnn_input, outputs=y)

combined = concatenate([mlp_branch.output, cnn_branch.output])

final_output = Dense(1, activation="linear")(combined)

model = Model(inputs=[mlp_branch.input, cnn_branch.input], outputs=final_output)
model.compile(loss="mean_absolute_error", optimizer=Adam(learning_rate=1e-3))

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │        448 │ input_layer_3[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 32,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │      4,640 │ max_pooling2d_2[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 16, 16,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 16)        │        176 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 8192)      │          0 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 4)         │         68 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 4)         │     32,772 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 8)         │          0 │ dense_5[0][0],    │
│ (Concatenate)       │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │          9 │ concatenate_1[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,113 (148.88 KB)

 Trainable params: 38,113 (148.88 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("[INFO] Training multimodal model...")
model.fit(
    x=[trainTab, trainImg], y=trainY,
    validation_data=([testTab, testImg], testY),
    epochs=50, batch_size=8
)
predictions = model.predict([testTab, testImg]).flatten()

mae = mean_absolute_error(testY, predictions)
rmse = np.sqrt(mean_squared_error(testY, predictions))

print("\n--- Final Evaluation Metrics ---")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")

[INFO] Training multimodal model...
Epoch 1/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - loss: 524244.7812 - val_loss: 560236.5000
Epoch 2/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 522239.0625 - val_loss: 554186.6875
Epoch 3/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 504662.7812 - val_loss: 517926.6250
Epoch 4/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 444490.3438 - val_loss: 432349.1875
Epoch 5/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 376003.5312 - val_loss: 347378.9062
Epoch 6/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 324153.4688 - val_loss: 292100.5000
Epoch 7/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 307298.8750 - val_loss: 278334.4062
Epoch 8/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 305521.7812 - val_loss: 279375.7500
Epoch 9/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 305753.5938 - val_loss: 275821.0625
Epoch 10/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 305584.2812 - val_loss: 277294.9375
Epoch 11/50
34/34 ━━━━